# Prepare dates, features and next-day answers

Unit IV covers feature engineering, missing values and splitting.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.data_preparation import load_city_data, build_features, build_targets
daily = load_city_data()


In [2]:
features = build_features(daily)
targets = build_targets(daily)
print(features.shape)
print(features.columns.tolist())

(2002, 29)
['PM2.5_today', 'PM2.5_lag_1', 'PM2.5_lag_2', 'PM2.5_lag_3', 'PM2.5_lag_7', 'PM10_lag_1', 'PM10_lag_2', 'PM10_lag_3', 'PM10_lag_7', 'NO2_lag_1', 'NO2_lag_2', 'NO2_lag_3', 'NO2_lag_7', 'CO_lag_1', 'CO_lag_2', 'CO_lag_3', 'CO_lag_7', 'O3_lag_1', 'O3_lag_2', 'O3_lag_3', 'O3_lag_7', 'PM2.5_mean_3', 'PM2.5_std_3', 'PM2.5_mean_7', 'PM2.5_std_7', 'month', 'day_of_week', 'day_of_year_sin', 'day_of_year_cos']


At origin day t, shift(1) brings yesterday into today's input row. shift(-1) brings tomorrow's observed answer into the target row. Rolling windows use shift(1) before rolling, ending at t-1. PM2.5_today explicitly uses the reading already available on day t. Tomorrow's calendar features are also known. The targets never enter the feature table.

In [3]:
preview = features[["PM2.5_today", "PM2.5_lag_1", "PM2.5_mean_3"]].join(targets[["target_date", "PM2.5_tomorrow", "AQI_tomorrow"]])
print(preview.head().to_string())

            PM2.5_today  PM2.5_lag_1  PM2.5_mean_3 target_date  PM2.5_tomorrow AQI_tomorrow
Date                                                                                       
2015-01-08       229.90       217.87    171.350000  2015-01-09          201.66    Very Poor
2015-01-09       201.66       229.90    199.116667  2015-01-10          221.02    Very Poor
2015-01-10       221.02       201.66    216.476667  2015-01-11          205.41    Very Poor
2015-01-11       205.41       221.02    217.526667  2015-01-12          212.41    Very Poor
2015-01-12       212.41       205.41    209.363333  2015-01-13          197.61    Very Poor


In [4]:
origin = features.index[0]
yesterday = origin - pd.Timedelta(days=1)
tomorrow = origin + pd.Timedelta(days=1)
print("Origin:", origin)
print("Lag one:", features.loc[origin, "PM2.5_lag_1"], "Original:", daily.loc[yesterday, "PM2.5"])
print("Tomorrow target:", targets.loc[origin, "PM2.5_tomorrow"], "Original:", daily.loc[tomorrow, "PM2.5"])
assert features.loc[origin, "PM2.5_lag_1"] == daily.loc[yesterday, "PM2.5"]
assert targets.loc[origin, "PM2.5_tomorrow"] == daily.loc[tomorrow, "PM2.5"]

Origin: 2015-01-08 00:00:00
Lag one: 217.87 Original: 217.87
Tomorrow target: 201.66 Original: 201.66


Train targets are before 2020-01-01. Test targets are from that date onwards. The first seven origins lack complete lag history and are excluded. Missing observed targets are dropped separately for regression and classification. Pipeline median imputation and scaling are fitted within each training fold. Class weights handle unequal class frequencies. High pollutant values are retained. The raw CSV remains the source.